# SQL Server Query Store and SQL Agent to Azure Monitor (OpenTelemetry)

Guided deployment. Each code cell is PowerShell and is labelled **Workstation** (PowerShell 7 + Azure CLI) or **VM** (elevated Windows PowerShell on the SQL Server VM).

To run cells directly, open this notebook in VS Code with the Jupyter extension and a PowerShell kernel. Otherwise, treat it as a checklist and paste each cell into the right shell. Full details are in `README.md`.

## 0. Configure

Edit `deploy.config.psd1` in this folder, then check it loads.

**Workstation**

In [ ]:
$cfg = Import-PowerShellDataFile ./deploy.config.psd1
$cfg.GetEnumerator() | Sort-Object Name | Format-Table -AutoSize

## 1. Application Insights with OTLP (portal, one time)

1. Run the next cell until the feature shows `Registered`.
2. In the portal, create an **Application Insights** resource with the name from the config, in the VM's region, with **Enable OTLP support** checked on the Basics tab.

**Workstation**

In [ ]:
az account set --subscription $cfg.SubscriptionId
az feature register --name OtlpApplicationInsights --namespace Microsoft.Insights
az feature show --name OtlpApplicationInsights --namespace Microsoft.Insights --query properties.state -o tsv

## 2. Azure setup

Finds the OTLP DCR and ApplicationId, installs or upgrades AMA, associates the DCR with the VM and writes `azure-outputs.json`.

**Workstation**

In [ ]:
./01-Setup-Azure.ps1
Get-Content ./azure-outputs.json

## 3. Copy the folder to the VM

Copy this whole folder, including `azure-outputs.json`, to `C:\otel\` on the VM (for example by Remote Desktop file copy). The remaining VM cells assume that path.

## 4. SQL setup

Creates the Collector login, grants permissions (including `VIEW DEFINITION` and msdb read access), and configures Query Store.

**VM**

In [ ]:
Set-ExecutionPolicy -Scope Process Bypass -Force
cd C:\otel
.\02-Setup-Sql.ps1

## 5. Wait for AMA's OTLP ports

AMA opens `127.0.0.1:4317` (metrics) and `127.0.0.1:4319` (logs) up to about 15 minutes after the DCR association.

**VM**

In [ ]:
Get-NetTCPConnection -LocalPort 4317,4319 -State Listen -ErrorAction SilentlyContinue |
  Select-Object LocalAddress, LocalPort, @{n='Process';e={(Get-Process -Id $_.OwningProcess).ProcessName}}

## 6. Install the Collector

Generates the config, test-runs it, and installs the Windows service. Logs turn on automatically when 4319 is listening; re-run this cell later if it installed metrics only.

**VM**

In [ ]:
cd C:\otel
.\03-Install-Collector.ps1

## 7. Optional: generate workload

Creates four demo Agent jobs and runs the AdventureWorks2019 procedure load for 20 minutes.

**VM**

In [ ]:
cd C:\otel\tools
sqlcmd -S localhost -E -C -i .\agent-demo-jobs.sql
.\Run-AwLoad.ps1 -Sessions 4

## 8. Verify the VM side

Every line should be `[ OK ]`. Data for a database appears after its first Query Store interval with workload closes.

**VM**

In [ ]:
cd C:\otel
.\04-Test-Pipeline.ps1

## 9. Verify in Azure

Shows new tables, `OTelLogs` rows by source, Query Store object types, and Agent job outcomes, plus PromQL to try in the Azure Monitor workspace.

**Workstation**

In [ ]:
./05-Verify-Azure.ps1

## 10. Import the dashboard

In the portal: **Application Insights > Dashboards with Grafana > New > Import**, upload `dashboards/sql-otel-grafana.json`, then choose the Azure Monitor data source.

To target a different Log Analytics workspace, regenerate it first.

**Workstation**

In [ ]:
$ws = (Get-Content ./azure-outputs.json -Raw | ConvertFrom-Json).LogAnalyticsResourceId
python ./dashboards/build_dashboard.py $ws

## Quick fixes

| Symptom | Fix |
| --- | --- |
| Logs sent but not in Log Analytics | `AI_APP_ID` must be the `ApplicationId`, not the InstrumentationKey |
| `[::1]` connection refused warnings | Exporters must use `127.0.0.1` |
| Nothing in your workspace | Query `LogAnalyticsWorkspaceId` from `azure-outputs.json` |
| Procedure panels empty | Wait for a closed interval with workload; see `04-Test-Pipeline.ps1` |

See `README.md` for the full troubleshooting table.